# HPC Module 1 — OpenMP debugging

This workshop has two modules: OpenMP and MPI. The organiser prepares
their shared environment once. Start here and move to `HPC/MPI/`
only when the organiser asks; no reset or reinstallation is needed.

## Learning outcomes

By completing these activities, you will practise creating a parallel
region, checking loop coverage, identifying shared-memory data races,
and correcting them with a reduction or an atomic update.

## How to work

For each exercise: inspect the C code, predict its behaviour, run it,
explain the bug, edit and save your correction, then test again.
Compare with the reference solution only after making your own attempt.

**Run cells individually, in order. Do not use Run All for your first
attempt:** later cells display the reference solutions.

Use the JupyterLab text editor to change each exercise's `code.c`.
This notebook does not automatically apply corrections. Viewing a
solution never overwrites your file.

Basic familiarity with C loops, arrays and variables is sufficient.
The OpenMP concepts are introduced in the activities below.


## 1. Check the prepared environment

Open this notebook from the `HPC/OpenMP/` directory.
The next cell checks the existing compiler and adds the local helper
directory to this notebook process's PATH. It installs nothing.

This PATH change also applies to commands launched by this notebook.
It does not configure separately opened terminals.

If the check fails, report it to the organiser. Do not install another
compiler or recreate the shared environment yourself.


In [ ]:
import os
from pathlib import Path
import shlex
import shutil

module_root = Path.cwd().resolve()
helper_directory = module_root / "helpers"
helper = helper_directory / "openmp_start"

if not helper.is_file() or not (module_root / "exercises").is_dir():
    raise RuntimeError(
        "Open this notebook with its working directory set to HPC/OpenMP/."
    )
if not os.access(helper, os.X_OK):
    raise RuntimeError("The exercise helper is not executable; ask the organiser.")

compiler = shlex.split(os.environ.get("CC", ""))
if not compiler or shutil.which(compiler[0]) is None:
    raise RuntimeError(
        "The prepared HPC compiler is unavailable; ask the organiser."
    )

current_path = os.environ.get("PATH", "")
if str(helper_directory) not in current_path.split(os.pathsep):
    os.environ["PATH"] = str(helper_directory) + os.pathsep + current_path

print("Module directory:", module_root)
print("Compiler command:", shlex.join(compiler))
print("Exercise helper:", shutil.which("openmp_start"))
print("No software was installed and no exercise file was changed.")


In [ ]:
!openmp_start --help


## 2. Understand the commands

- `openmp_start code1`: display your current C source.
- `openmp_start code1 --problem`: display the activity instructions.
- `openmp_start code1 --run --threads 2`: compile and run your current source.
- `openmp_start code1 --solution`: display the reference code and explanation.

`--repeat` repeats execution; `--timeout` limits each execution in seconds.
The helper rebuilds before running and never runs an old executable after
a failed build. Builds use temporary directories.

Examples below request at most two threads. Do not increase resource use
beyond the organiser's instructions.

A successful process exit does not prove algorithmic correctness.
In particular, one correct-looking result cannot rule out a data race.


## code1 — Parallel regions

A normal C block runs serially. An OpenMP parallel region creates
a team whose members each execute the block. Requesting two threads
does not automatically parallelise ordinary C code.


# code1 — Why is there only one thread?

## Learning goal

Understand that requesting threads does not create a parallel region.
Identify where an OpenMP parallel directive belongs.

## Intended behaviour

When run with `--threads 2`, the corrected program should enter a
parallel region and print one greeting from each participating thread.

With a two-thread team, expect two greetings: one from thread 0 and
one from thread 1. Their order is not guaranteed.

## Observe the supplied program

    openmp_start code1
    openmp_start code1 --run --threads 2

The supplied program prints one greeting reporting a team size of 1.
It exits successfully, but does not demonstrate the intended parallelism.

## Investigate

1. What do `omp_get_thread_num()` and `omp_get_num_threads()` report?
2. Is the greeting currently inside an OpenMP parallel region?
3. Does setting `OMP_NUM_THREADS` create a parallel region by itself?
4. Which directive would make all team members execute this block?

## Propose and test a correction

Edit `exercises/code1/code.c` using the JupyterLab text editor.
Keep the braces and greeting; add the directive you believe is missing.

Then save the file and run:

    openmp_start code1 --run --threads 2
    openmp_start code1 --run --threads 4
    openmp_start code1 --run --threads 2 --repeat 3

Compare the reported team size and greeting count with your prediction.
Check correctness, not the order of the greetings.

## Compare with the reference

After making and testing your own correction:

    openmp_start code1 --solution

This displays the reference code and explanation without changing
your working file.

## Reflection

Explain the difference between requesting a team size and creating
a parallel region. Explain why changing greeting order is not a bug.

## Runtime qualification

The runner requests the thread count and disables dynamic adjustment.
The actual team size still depends on the program and runtime resources.
If the corrected program cannot create the requested team, ask the
organiser rather than reinstalling the environment.


### Inspect and predict

Run the next cell to inspect your current source. Before executing
the program, write your prediction and reasoning in your notes.


In [ ]:
!openmp_start code1


In [ ]:
!openmp_start code1 --run --threads 1


In [ ]:
!openmp_start code1 --run --threads 2


### Make your correction

Open **`exercises/code1/code.c`** in the JupyterLab text editor.
Explain your proposed change, edit the file, and save it.

Run the next cell to rebuild and test your saved correction.
The runner uses your working source, not the reference solution.

For `code2`, the original exit status 1 is an intentional failed
coverage check. For race exercises, a MATCH is not proof that the
original program is safe.


In [ ]:
!openmp_start code1 --run --threads 2 --repeat 3


### Optional reference comparison — after your attempt

The next cell reveals the reference solution and explanation.
Skip it until you have proposed and tested your own correction.
It displays information only; it does not change your working source.


In [ ]:
!openmp_start code1 --solution


### Reflection

Record the bug, your correction, and the evidence you used.
Answer the reflection questions in the activity description.
Explain whether your correction works with both one and two threads.


## code2 — Worksharing and loop coverage

A parallel-for construct distributes the specified loop iterations
among team members. It does not correct the loop bounds. Distinct
iterations can safely write distinct array elements.


# code2 — Why is the last array element unchanged?

## Learning goal

Separate loop coverage from parallel execution.
Understand the bounds of an OpenMP worksharing loop.

## Intended behaviour

Every array element should contain the square of its index.
The final coverage check should report PASS.

## Observe

    openmp_start code2
    openmp_start code2 --run --threads 2

The supplied program leaves the last element at -1.
It reports FAIL and returns exit status 1. This is an
intentional exercise failure, not an installation failure.

## Investigate and correct

1. List the valid indices of an array with 8 elements.
2. List the indices visited by the parallel loop.
3. Explain why requesting more threads cannot repair its bounds.
4. Edit and save `exercises/code2/code.c` to cover every index.

Test your correction with one and two requested threads:

    openmp_start code2 --run --threads 1
    openmp_start code2 --run --threads 2

## Compare and reflect

    openmp_start code2 --solution

Explain why concurrent writes are safe here: different loop
iterations write to different array elements.
Explain why the checks after the loop see completed writes.


### Inspect and predict

Run the next cell to inspect your current source. Before executing
the program, write your prediction and reasoning in your notes.


In [ ]:
!openmp_start code2


In [ ]:
!openmp_start code2 --run --threads 1


In [ ]:
!openmp_start code2 --run --threads 2


### Make your correction

Open **`exercises/code2/code.c`** in the JupyterLab text editor.
Explain your proposed change, edit the file, and save it.

Run the next cell to rebuild and test your saved correction.
The runner uses your working source, not the reference solution.

For `code2`, the original exit status 1 is an intentional failed
coverage check. For race exercises, a MATCH is not proof that the
original program is safe.


In [ ]:
!openmp_start code2 --run --threads 2 --repeat 3


### Optional reference comparison — after your attempt

The next cell reveals the reference solution and explanation.
Skip it until you have proposed and tested your own correction.
It displays information only; it does not change your working source.


In [ ]:
!openmp_start code2 --solution


### Reflection

Record the bug, your correction, and the evidence you used.
Answer the reflection questions in the activity description.
Explain whether your correction works with both one and two threads.


## code3 — Shared sums and reductions

Threads can interfere when updating the same shared variable.
A reduction combines independent partial results using a specified
operation, avoiding unsynchronised updates to the shared accumulator.


# code3 — Why does a parallel sum sometimes lose updates?

## Learning goal

Identify a data race and use an OpenMP reduction.

## Intended behaviour

Add the integers from 1 through 10000.
The expected sum is 50005000.

## Observe

    openmp_start code3
    openmp_start code3 --run --threads 1
    openmp_start code3 --run --threads 2 --repeat 5

Compare the printed result with the expected sum.
A mismatch may appear, but it is not guaranteed.

## Investigate and correct

1. Is `sum` shared by the threads?
2. Does `sum += i` perform an indivisible update?
3. Explain how two updates can interfere.
4. Find the OpenMP clause for combining partial sums.
5. Edit and save `exercises/code3/code.c`, then repeat the runs.

## Important limitation

The original program contains a data race. Its result is not
reliably defined; a MATCH in any number of runs does not
establish correctness. Do not treat an observed wrong value
as the only evidence of the bug.

The program deliberately returns exit status 0 so the runner
can repeat observations. Judge the code and printed comparison,
not exit status alone.

## Compare and reflect

    openmp_start code3 --solution

Explain what each thread accumulates and how OpenMP combines
those partial results.


### Inspect and predict

Run the next cell to inspect your current source. Before executing
the program, write your prediction and reasoning in your notes.


In [ ]:
!openmp_start code3


In [ ]:
!openmp_start code3 --run --threads 1


In [ ]:
!openmp_start code3 --run --threads 2


### Make your correction

Open **`exercises/code3/code.c`** in the JupyterLab text editor.
Explain your proposed change, edit the file, and save it.

Run the next cell to rebuild and test your saved correction.
The runner uses your working source, not the reference solution.

For `code2`, the original exit status 1 is an intentional failed
coverage check. For race exercises, a MATCH is not proof that the
original program is safe.


In [ ]:
!openmp_start code3 --run --threads 2 --repeat 3


### Optional reference comparison — after your attempt

The next cell reveals the reference solution and explanation.
Skip it until you have proposed and tested your own correction.
It displays information only; it does not change your working source.


In [ ]:
!openmp_start code3 --solution


### Reflection

Record the bug, your correction, and the evidence you used.
Answer the reflection questions in the activity description.
Explain whether your correction works with both one and two threads.


## code4 — Atomic updates

Incrementing a shared counter involves a read, an addition and a write.
An atomic update protects this single operation from interfering
updates. It does not make an arbitrary block of statements atomic.


# code4 — Why does the shared counter miss increments?

## Learning goal

Recognise a read-modify-write race and protect a single update
with an OpenMP atomic directive.

## Intended behaviour

Perform 100000 increments. The final counter should be 100000.

## Observe

    openmp_start code4
    openmp_start code4 --run --threads 1
    openmp_start code4 --run --threads 2 --repeat 5

A multi-threaded run may show a mismatch. A mismatch is not
guaranteed: the original code has a data race, and a MATCH
does not prove correctness.

## Investigate and correct

1. Which variable is shared?
2. Explain the read, addition and write involved in an increment.
3. Explain how two increments could lose an update.
4. Find the directive that protects one shared-memory update.
5. Edit and save `exercises/code4/code.c`, then repeat the runs.

The program returns exit status 0 to allow repeated observations.
Inspect the printed comparison and reason about synchronisation.

## Compare and reflect

    openmp_start code4 --solution

Explain why the atomic directive applies to the increment,
not the entire loop. Could a reduction also solve this example?
Which approach would you choose if only the final total matters?


### Inspect and predict

Run the next cell to inspect your current source. Before executing
the program, write your prediction and reasoning in your notes.


In [ ]:
!openmp_start code4


In [ ]:
!openmp_start code4 --run --threads 1


In [ ]:
!openmp_start code4 --run --threads 2


### Make your correction

Open **`exercises/code4/code.c`** in the JupyterLab text editor.
Explain your proposed change, edit the file, and save it.

Run the next cell to rebuild and test your saved correction.
The runner uses your working source, not the reference solution.

For `code2`, the original exit status 1 is an intentional failed
coverage check. For race exercises, a MATCH is not proof that the
original program is safe.


In [ ]:
!openmp_start code4 --run --threads 2 --repeat 3


### Optional reference comparison — after your attempt

The next cell reveals the reference solution and explanation.
Skip it until you have proposed and tested your own correction.
It displays information only; it does not change your working source.


In [ ]:
!openmp_start code4 --solution


### Reflection

Record the bug, your correction, and the evidence you used.
Answer the reflection questions in the activity description.
Explain whether your correction works with both one and two threads.


## Final review

1. Why does requesting a thread count not create a parallel region?
2. Why can more threads not repair an incorrect loop bound?
3. Why is a correct-looking output insufficient to exclude a data race?
4. How do a reduction and an atomic update differ?
5. Why is exit status alone insufficient to assess these exercises?

Keep your saved corrections and notes. Moving to the MPI module requires
no environment transition: wait for the organiser's instruction.

## Material and licence

These debugging exercises were newly authored for this workshop.
The C files carry Apache-2.0 SPDX identifiers. All new workshop material is distributed under Apache-2.0.
See `../LICENSE` and `../PROVENANCE.md` for licence and provenance.
